In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene
import statsmodels.stats.multitest as smm

import sys
sys.path.append('../../../')
sys.path.append('../../')
from utilis import *
from settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

# Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
outliers = []  # Participant exclusions are maintained privately.

print(df.shape) 
df = df.drop(df[df['ID'].isin(outliers)].index)
print(df.shape) 

In [ ]:
(df['p-tau217'] > HIGH_TAU_217_THR).value_counts()

# Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
# removing samples with moca below the minimum threshold of 19
df = df[df[MOCA_COL] > MIN_MOCA_SCORE]
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])

# Filter For Females

In [ ]:
df_high_vs_low = df_high_vs_low[df_high_vs_low['Gender'] == 'm']


In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

## Cognitive Tests

In [ ]:
COGNITIVE_SAVE_DIR = SAVE_DIR + 'cognitive/'

Filter cognitive damaged subjects

In [ ]:
df_high_vs_low_no_mci = df_high_vs_low[df_high_vs_low['MCI'] == False]
df_high_vs_low_no_mci[TAU_CAT].value_counts()

### Executive

In [ ]:
EXECTUTIVE = 'Executive'

df_high_vs_low_exe = df_high_vs_low_no_mci[df_high_vs_low_no_mci[EXECTUTIVE].notna()]
graph_colors = [healthy_color, risk_color]
kdeplot_high_vs_low(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, graph_colors, 'Executive vs Tau 217 category',COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_exe, EXECTUTIVE, TAU_CAT, 4, graph_colors, 'Executive vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Executive_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'Low'][EXECTUTIVE]
group2 = df_high_vs_low_exe[df_high_vs_low_exe[TAU_CAT] == 'High'][EXECTUTIVE]

p_executive = stat_test(data_type, group1, group2)

### Spatial

In [ ]:
SPATIAL = 'Spatial'
df_high_vs_low_spatial = df_high_vs_low_no_mci[df_high_vs_low_no_mci[SPATIAL].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_spatial, SPATIAL, TAU_CAT, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_spatial, SPATIAL, TAU_CAT, 4, graph_colors, 'Spatial vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Spatial_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'Low'][SPATIAL]
group2 = df_high_vs_low_spatial[df_high_vs_low_spatial[TAU_CAT] == 'High'][SPATIAL]

p_spatial = stat_test(data_type, group1, group2)

### Attention_and_concentration

In [ ]:
A_C = 'Attention_and_concentration'
df_high_vs_low_ac = df_high_vs_low_no_mci[df_high_vs_low_no_mci[A_C].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_ac, A_C, TAU_CAT, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_ac, A_C, TAU_CAT, 5, graph_colors, 'Attention/Concentration vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Attention_Concentration_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Ordinal'
group1 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'Low'][A_C]
group2 = df_high_vs_low_ac[df_high_vs_low_ac[TAU_CAT] == 'High'][A_C]

p_attention = stat_test(data_type, group1, group2)

### Memory

In [ ]:
MEMORY = 'Memory'
df_high_vs_low_memory = df_high_vs_low_no_mci[df_high_vs_low_no_mci[MEMORY].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_memory, MEMORY, TAU_CAT, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_memory, MEMORY, TAU_CAT, 5, graph_colors, 'Memory vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Memory_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'Low'][MEMORY]
group2 = df_high_vs_low_memory[df_high_vs_low_memory[TAU_CAT] == 'High'][MEMORY]

p_memory = stat_test(data_type, group1, group2)

### Language

In [ ]:
LANGUAGE = 'Language'
df_high_vs_low_language = df_high_vs_low_no_mci[df_high_vs_low_no_mci[LANGUAGE].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_language, LANGUAGE, TAU_CAT, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_language, LANGUAGE, TAU_CAT, 5, graph_colors, 'Language vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Language_vs_Tau_217_category_hist.pdf')

In [ ]:
# statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'Low'][LANGUAGE]
group2 = df_high_vs_low_language[df_high_vs_low_language[TAU_CAT] == 'High'][LANGUAGE]

p_language = stat_test(data_type, group1, group2)

### Orientation

In [ ]:
ORIENTATION = 'Orientation'

df_high_vs_low_orientation = df_high_vs_low_no_mci[df_high_vs_low_no_mci[ORIENTATION].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_orientation, ORIENTATION, TAU_CAT, 6, graph_colors, 'Orientation vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Orientation_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_orientation
group2 = [TAU_CAT, ORIENTATION]

p_orientation = stat_test(data_type, group1, group2) # Too few samples for the test

### Frontal memory score (#1)

In [ ]:
FMC = 'Frontal memory score (#1)'

df_high_vs_low_fmc = df_high_vs_low_no_mci[df_high_vs_low_no_mci[FMC].notna()]
df_high_vs_low_fmc[FMC] = df_high_vs_low_fmc[FMC].astype(float)
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_fmc, FMC, TAU_CAT, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_fmc, FMC, TAU_CAT, 4, graph_colors, 'Frontal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Frontal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'Low'][FMC]
group2 = df_high_vs_low_fmc[df_high_vs_low_fmc[TAU_CAT] == 'High'][FMC]

p_frontal_memory = stat_test(data_type, group1, group2)

### Hipocampal memory score (#2)

In [ ]:
HMC = 'Hipocampal memory score (#2)'

df_high_vs_low_hmc = df_high_vs_low_no_mci[df_high_vs_low_no_mci[HMC].notna()]
graph_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_hmc, HMC, TAU_CAT, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_kde.pdf')
plot_histplot(df_high_vs_low_hmc, HMC, TAU_CAT, 5, graph_colors, 'Hipocampal Memory Score vs Tau 217 category', COGNITIVE_SAVE_DIR + 'Hipocampal_Memory_Score_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Ordinal'
group1 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'Low'][HMC]
group2 = df_high_vs_low_hmc[df_high_vs_low_hmc[TAU_CAT] == 'High'][HMC]

p_hipocampal_memory = stat_test(data_type, group1, group2)

# Statistical Correction
I'm using BH-FDR

In [ ]:
ALPHA = 0.05
METHOD = 'fdr_bh'

In [ ]:
# Statistical Correction
p_values = [p_attention, p_executive, p_spatial, p_memory, p_language, p_orientation, p_frontal_memory, p_hipocampal_memory]
risk_factors = ['Attention and concentration', 'Executive', 'Spatial', 'Memory', 'Language', 'Orientation', 'Frontal memory score', 'Hipocampal memory score']
p_values = [p for p in p_values if p is not None]
reject, p_values_corrected, _, _ = smm.multipletests(p_values, alpha=ALPHA, method=METHOD)

for i, p in enumerate(p_values):
    if p is not None:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

print("\nSignificant features:")
for i, p in enumerate(p_values):
    if p < ALPHA:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

In [ ]:
HIGH_TAU_217_THR

# Results
### 0.53
* No features where signifcant even before correction

### 0.449
* No features where signifcant even before correction